In [1]:
import pandas as pd
import re
import nltk
from IPython.display import display

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')


file_path = "data_sampling20_komen.csv"
df = pd.read_csv(file_path)

print("data sebelum preprocessing")
display(df[['review_detail']].head(3))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


data sebelum preprocessing


,review_detail
0,"I enjoyed the first season, but I must say I t..."
1,I know Iceland is a small country and police d...
2,"Except K K , no other actor looks comfortable ..."


In [2]:
def case_folding(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['case_folded'] = df['review_detail'].apply(case_folding)

print("data setelah case-folding, menghapus karakter selain a-z, dan menghapus spasi berlebih")
display(df[['review_detail', 'case_folded']].head(3))

data setelah case-folding, menghapus karakter selain a-z, dan menghapus spasi berlebih


,review_detail,case_folded
0,"I enjoyed the first season, but I must say I t...",i enjoyed the first season but i must say i th...
1,I know Iceland is a small country and police d...,i know iceland is a small country and police d...
2,"Except K K , no other actor looks comfortable ...",except k k no other actor looks comfortable in...


In [3]:
from nltk.tokenize import word_tokenize

df['tokenized'] = df['case_folded'].apply(word_tokenize)

print("data setelah tokenization")
display(df[['case_folded', 'tokenized']].head(3))

data setelah tokenization


,case_folded,tokenized
0,i enjoyed the first season but i must say i th...,"[i, enjoyed, the, first, season, but, i, must,..."
1,i know iceland is a small country and police d...,"[i, know, iceland, is, a, small, country, and,..."
2,except k k no other actor looks comfortable in...,"[except, k, k, no, other, actor, looks, comfor..."


In [4]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['no_stopwords'] = df['tokenized'].apply(remove_stopwords)

print("data setelah stopwords removal")
display(df[['tokenized', 'no_stopwords']].head(3))

data setelah stopwords removal


,tokenized,no_stopwords
0,"[i, enjoyed, the, first, season, but, i, must,...","[enjoyed, first, season, must, say, think, sea..."
1,"[i, know, iceland, is, a, small, country, and,...","[know, iceland, small, country, police, things..."
2,"[except, k, k, no, other, actor, looks, comfor...","[except, k, k, actor, looks, comfortable, acti..."


In [5]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['lemmatized'] = df['no_stopwords'].apply(lemmatize_text)

df['clean_review'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print("Data setelah lemmatization")
display(df[['no_stopwords', 'clean_review']].head(3))

Data setelah lemmatization


,no_stopwords,clean_review
0,"[enjoyed, first, season, must, say, think, sea...",enjoyed first season must say think season eve...
1,"[know, iceland, small, country, police, things...",know iceland small country police thing bit di...
2,"[except, k, k, actor, looks, comfortable, acti...",except k k actor look comfortable acting fight...


In [7]:
df.head()

,review_detail,case_folded,tokenized,no_stopwords,lemmatized,clean_review
0,"I enjoyed the first season, but I must say I t...",i enjoyed the first season but i must say i th...,"[i, enjoyed, the, first, season, but, i, must,...","[enjoyed, first, season, must, say, think, sea...","[enjoyed, first, season, must, say, think, sea...",enjoyed first season must say think season eve...
1,I know Iceland is a small country and police d...,i know iceland is a small country and police d...,"[i, know, iceland, is, a, small, country, and,...","[know, iceland, small, country, police, things...","[know, iceland, small, country, police, thing,...",know iceland small country police thing bit di...
2,"Except K K , no other actor looks comfortable ...",except k k no other actor looks comfortable in...,"[except, k, k, no, other, actor, looks, comfor...","[except, k, k, actor, looks, comfortable, acti...","[except, k, k, actor, look, comfortable, actin...",except k k actor look comfortable acting fight...
3,I'm guessing that as a 62 year old white woman...,im guessing that as a year old white woman im ...,"[im, guessing, that, as, a, year, old, white, ...","[im, guessing, year, old, white, woman, im, ta...","[im, guessing, year, old, white, woman, im, ta...",im guessing year old white woman im target dem...
4,Here's the truth. There's not much to this mov...,heres the truth theres not much to this movie ...,"[heres, the, truth, theres, not, much, to, thi...","[heres, truth, theres, much, movie, sucked, hi...","[here, truth, there, much, movie, sucked, high...",here truth there much movie sucked high rating...


In [6]:
pd.DataFrame(df).to_csv("data_hasil_preprocessing.csv",index=False)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Mengatasi missing value
df['clean_review'] = df['clean_review'].fillna('')

# Inisialisasi TF-IDF dengan beberapa pembatasan supaya tidak berat
tfidf_vectorizer = TfidfVectorizer(
    min_df=5,         
    max_df=0.8,         
    max_features=10000  
)

# Fit dan Transform data teks menjadi matriks angka
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_review'])

print("Informasi data")
print(f"Jumlah Dokumen (Baris): {tfidf_matrix.shape[0]}")
print(f"Jumlah Fitur/Kata (Kolom): {tfidf_matrix.shape[1]}")
print("-" * 40)

feature_names = tfidf_vectorizer.get_feature_names_out()
df_tfidf_sample = pd.DataFrame(tfidf_matrix[:5].toarray(), columns=feature_names)

print("\ndata setelah tf-idf vectorizer")
display(df_tfidf_sample.iloc[:, :10])

Informasi data
Jumlah Dokumen (Baris): 295357
Jumlah Fitur/Kata (Kolom): 10000
----------------------------------------

data setelah tf-idf vectorizer


,aamir,aaron,ab,abandon,abandoned,abbey,abbott,abby,abc,abducted
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
import scipy.sparse
import pickle

path_clean_csv = "E:\Kuliah\DatMin\PROJEK\data_sampling30_clean.csv"
path_tfidf_matrix = "E:\Kuliah\DatMin\PROJEK\\tfidf_matrix_data_sampling30.npz"
path_tfidf_vectorizer = "E:\Kuliah\DatMin\PROJEK\\tfidf_vectorizer_data_sampling30.pkl"

df_final = df[['clean_review']]
df_final.to_csv(path_clean_csv, index=False)
print(f"{path_clean_csv}")

# Menyimpan ke NPZ 
scipy.sparse.save_npz(path_tfidf_matrix, tfidf_matrix)
print(f"{path_tfidf_matrix}")

# TAHAP 3: Menyimpan Kamus/Model TF-IDF ke PKL(buat modelling)
with open(path_tfidf_vectorizer, 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f"{path_tfidf_vectorizer}")

E:\Kuliah\DatMin\PROJEK\data_sampling30_clean.csv
E:\Kuliah\DatMin\PROJEK\tfidf_matrix_data_sampling30.npz
E:\Kuliah\DatMin\PROJEK\tfidf_vectorizer_data_sampling30.pkl
